# Befolkningsvekst — Folkeregister

Visualiserer befolkningsutvikling, fødselstall og dødsfall over simuleringsperioden per kommune og totalt.

**Kilde:** `folkeregister.population_stats` (Gold-lag)  
**Relatert issue:** US E-1 / #93

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# DuckDB med MinIO/Delta
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta;  LOAD delta;")

_raw = os.environ.get("MINIO_ENDPOINT", "http://minio.slettix-analytics.svc.cluster.local:9000")
_endpoint = _raw.replace("http://", "").replace("https://", "")

con.execute(f"""
    CREATE OR REPLACE SECRET minio (
        TYPE      S3,
        KEY_ID    '{os.environ.get("MINIO_ACCESS_KEY", "admin")}',
        SECRET    '{os.environ.get("MINIO_SECRET_KEY", "changeme")}',
        ENDPOINT  '{_endpoint}',
        URL_STYLE 'path',
        USE_SSL   false
    );
""")

POPULATION_PATH = "s3://gold/folkeregister/population_stats"
print("DuckDB klar.", duckdb.__version__)

In [ ]:
# Last inn data
df = con.execute(f"SELECT * FROM delta_scan('{POPULATION_PATH}')").df()
print(f"{len(df):,} rader | {df['reference_year'].nunique()} år | {df['municipality_code'].nunique()} kommuner")
df.head()

## 1. Total befolkningsutvikling over tid

In [ ]:
totals = df.groupby("reference_year").agg(
    population=("population_total", "sum"),
    births=("births", "sum"),
    deaths=("deaths", "sum"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(totals["reference_year"], totals["population"], marker="o", linewidth=2, color="steelblue", label="Total befolkning")
ax.fill_between(totals["reference_year"], totals["population"], alpha=0.1, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.set_xlabel("År")
ax.set_ylabel("Befolkning")
ax.set_title("Total befolkning over simuleringsperioden", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Fødsler vs. dødsfall per år

In [ ]:
years = totals["reference_year"]
x = range(len(years))
width = 0.4

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar([i - width/2 for i in x], totals["births"],  width=width, color="#2ecc71", label="Fødsler")
ax.bar([i + width/2 for i in x], totals["deaths"],  width=width, color="#e74c3c", label="Dødsfall")
ax.set_xticks(list(x))
ax.set_xticklabels(years, rotation=45, ha="right")
ax.set_ylabel("Antall")
ax.set_title("Fødsler vs. dødsfall per år (totalt)", fontweight="bold")
ax.legend()

# Naturlig vekst-linje
natural_growth = totals["births"] - totals["deaths"]
ax2 = ax.twinx()
ax2.plot(list(x), natural_growth, color="navy", linestyle="--", marker="s", markersize=4, label="Naturlig vekst")
ax2.set_ylabel("Naturlig vekst", color="navy")
ax2.tick_params(axis="y", labelcolor="navy")
ax2.axhline(0, color="navy", linewidth=0.5, alpha=0.4)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 3. Befolkningsutvikling per kommune — topp 8

In [ ]:
# Velg kommunene med høyest gjennomsnittlig befolkning
top_kom = (
    df.groupby("municipality_name")["population_total"]
    .mean()
    .nlargest(8)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.get_cmap("tab10")
for i, kom in enumerate(top_kom):
    sub = df[df["municipality_name"] == kom].sort_values("reference_year")
    ax.plot(sub["reference_year"], sub["population_total"], marker=".", color=cmap(i), label=kom)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xlabel("År")
ax.set_ylabel("Befolkning")
ax.set_title("Befolkningsutvikling — topp 8 kommuner", fontweight="bold")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Raskest voksende kommuner (siste år vs. første år)

In [ ]:
first_year = df["reference_year"].min()
last_year  = df["reference_year"].max()

pop_first = df[df["reference_year"] == first_year].set_index("municipality_name")["population_total"]
pop_last  = df[df["reference_year"] == last_year].set_index("municipality_name")["population_total"]

growth = ((pop_last - pop_first) / pop_first.replace(0, float("nan")) * 100).dropna()
growth_top = growth.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in growth_top]
ax.barh(growth_top.index, growth_top.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"Befolkningsvekst {first_year}→{last_year} (%)")
ax.set_title("Topp 15 kommuner — prosentvis befolkningsvekst", fontweight="bold")
plt.tight_layout()
plt.show()